# Construção da bridge CINE–CBO

Constrói `workspace.gold.bridge_cine_cbo`, relacionando rótulos CINE Brasil (`dim_cine_brasil`, 372 códigos) a ocupações CBO 2002 (`dim_cbo_ocupacao`, 2.778 códigos), a partir do CSV de correspondência fornecido pelo MEC. Enquanto o CSV não for recebido, o notebook encerra com status `AGUARDANDO_FONTE` — sem criar uma bridge vazia e sem gerar correspondências fictícias.

## Roteiro

1. **Configurações** — caminhos parametrizáveis (widgets), versão do mapeamento e tabelas de referência;
2. **Disponibilidade da fonte** — enquanto o CSV do MEC não constar no volume, o notebook encerra com status `AGUARDANDO_FONTE` **sem criar a bridge**;
3. **Leitura** — CSV com todas as colunas como STRING (preserva zeros à esquerda), separador configurável;
4. **Padronização** — `codigo_cine_rotulo` (trim + maiúsculas) e `codigo_cbo` (trim + 6 dígitos com zero à esquerda);
5. **Validações** — formato, nulos, pares duplicados e existência dos códigos nas dimensões, sem exclusões silenciosas;
6. **Cobertura** — indicadores por CINE e por CBO, incluindo códigos sem correspondência;
7. **Gravação idempotente** — só regrava quando a fonte ou a versão do mapeamento mudar;
8. **Validação pós-gravação** — unicidade dos pares e integridade com as dimensões.


In [0]:
from pyspark.sql import functions as F

# Parâmetros (widgets) — sobrescritos por jobs/agendamentos sem alterar o código
dbutils.widgets.text("caminho_diretorio_csv", "/Volumes/workspace/bronze/raw_cine_cbo", "Diretório do CSV do MEC")
dbutils.widgets.text("arquivo_csv", "", "Arquivo CSV (vazio = detectar automaticamente)")
dbutils.widgets.text("separador_csv", ";", "Separador do CSV")
dbutils.widgets.text("versao_mapeamento", "1.0.0", "Versão do mapeamento recebido")

# Resolução dos parâmetros
DIRETORIO_CSV = dbutils.widgets.get("caminho_diretorio_csv")
ARQUIVO_CSV = dbutils.widgets.get("arquivo_csv").strip()
SEPARADOR_CSV = dbutils.widgets.get("separador_csv")
VERSAO_MAPEAMENTO = dbutils.widgets.get("versao_mapeamento").strip()

# Tabelas envolvidas
DIM_CINE = "workspace.gold.dim_cine_brasil"
DIM_CBO = "workspace.gold.dim_cbo_ocupacao"
BRIDGE = "workspace.gold.bridge_cine_cbo"

# Metadados da bridge
FONTE_MAPEAMENTO = "MEC — CSV de correspondência CINE–CBO"

# Padrões validados contra as dimensões existentes:
# - 372/372 códigos CINE obedecem a 4 dígitos + letra + 2 dígitos
# - 2778/2778 códigos CBO têm 6 dígitos
PADRAO_CINE = r"^[0-9]{4}[A-Z][0-9]{2}$"
PADRAO_CBO = r"^[0-9]{6}$"

# Nomes de coluna aceitos no CSV (por ordem de preferência)
CANDIDATAS_COLUNA_CINE = ["codigo_cine_rotulo", "cine_rotulo", "codigo_cine", "cod_cine_rotulo", "rotulo_cine"]
CANDIDATAS_COLUNA_CBO = ["codigo_cbo", "cbo", "cbo2002", "codigo_cbo2002", "ocupacao_cbo"]

print("Diretório do CSV:", DIRETORIO_CSV)
print("Arquivo CSV:", ARQUIVO_CSV or "(detecção automática)")
print("Separador:", repr(SEPARADOR_CSV))
print("Versão do mapeamento:", VERSAO_MAPEAMENTO)
print("Tabela-alvo:", BRIDGE)

In [0]:
# Status de encerramento quando a fonte não está disponível
STATUS_AGUARDANDO = "AGUARDANDO_FONTE"

arquivos_csv = []

try:
    itens = dbutils.fs.ls(DIRETORIO_CSV)
    if ARQUIVO_CSV:
        arquivos_csv = [
            i.path for i in itens
            if i.name == ARQUIVO_CSV and not i.isDir()
        ]
    else:
        arquivos_csv = [
            i.path for i in itens
            if i.name.lower().endswith(".csv") and not i.isDir()
        ]
    conteudo_diretorio = [(i.name, i.size) for i in itens]
except Exception as e:
    conteudo_diretorio = []
    print(f"AVISO: não foi possível listar {DIRETORIO_CSV} ({e})")

if not arquivos_csv:
    print("=" * 60)
    print(f"STATUS: {STATUS_AGUARDANDO}")
    print(f"CSV de correspondência CINE–CBO não encontrado em: {DIRETORIO_CSV}")
    if conteudo_diretorio:
        print("Conteúdo atual do diretório:", conteudo_diretorio)
    print("A bridge NÃO será criada — nenhum dado fictício é gerado.")
    print("Coloque o CSV do MEC no volume acima e reexecute este notebook.")
    print("=" * 60)
    dbutils.notebook.exit(STATUS_AGUARDANDO)

if len(arquivos_csv) > 1:
    raise ValueError(
        f"Mais de um CSV encontrado em {DIRETORIO_CSV}: {arquivos_csv}. "
        "Especifique o parâmetro 'arquivo_csv' para evitar seleção arbitrária."
    )

CAMINHO_CSV = arquivos_csv[0]
print("CSV de correspondência encontrado:", CAMINHO_CSV)

In [0]:
# Leitura com todas as colunas como STRING — preserva zeros à esquerda
df_csv = (
    spark.read
    .option("header", True)
    .option("sep", SEPARADOR_CSV)
    .option("encoding", "UTF-8")
    .option("inferSchema", False)
    .csv(CAMINHO_CSV)
)

print("Schema lido (todas as colunas como STRING):")
df_csv.printSchema()
print(f"\nLinhas lidas: {df_csv.count()}")

# Identificação das colunas de correspondência (nomes podem variar no arquivo do MEC)
colunas_presentes = [c.strip().lower() for c in df_csv.columns]


def resolver_coluna(candidatas):
    for nome in candidatas:
        if nome in colunas_presentes:
            return df_csv.columns[colunas_presentes.index(nome)]
    return None


COLUNA_CINE = resolver_coluna(CANDIDATAS_COLUNA_CINE)
COLUNA_CBO = resolver_coluna(CANDIDATAS_COLUNA_CBO)

if COLUNA_CINE is None or COLUNA_CBO is None:
    raise ValueError(
        "Colunas de correspondência não identificadas no CSV.\n"
        f"Esperado algo como '{CANDIDATAS_COLUNA_CINE[0]}' e '{CANDIDATAS_COLUNA_CBO[0]}'.\n"
        f"Colunas encontradas: {df_csv.columns}"
    )

print(f"Coluna CINE: '{COLUNA_CINE}' | Coluna CBO: '{COLUNA_CBO}'")

display(df_csv.limit(20))

In [0]:
# Padronização: trim e maiúsculas no CINE; CBO com 6 dígitos (zero à esquerda)
df_mapeamento = df_csv.select(
    F.upper(F.trim(F.col(COLUNA_CINE))).alias("codigo_cine_rotulo"),
    F.lpad(F.trim(F.col(COLUNA_CBO)), 6, "0").alias("codigo_cbo"),
)

print("Padronização aplicada: trim + maiúsculas (CINE) e lpad de 6 dígitos (CBO).")
display(df_mapeamento.limit(20))

In [0]:
# 1) Nulos e formato — abortam a execução exibindo as linhas afetadas
resumo_formato = df_mapeamento.agg(
    F.count("*").alias("qtd_linhas"),
    F.sum(F.when(F.col("codigo_cine_rotulo").isNull(), 1).otherwise(0)).alias("cine_nulos"),
    F.sum(F.when(F.col("codigo_cbo").isNull(), 1).otherwise(0)).alias("cbo_nulos"),
    F.sum(F.when(~F.col("codigo_cine_rotulo").rlike(PADRAO_CINE), 1).otherwise(0)).alias("cine_fora_padrao"),
    F.sum(F.when(~F.col("codigo_cbo").rlike(PADRAO_CBO), 1).otherwise(0)).alias("cbo_fora_padrao"),
).collect()[0]

print("Linhas no CSV:", resumo_formato["qtd_linhas"])
print("CINE nulos/fora do padrão:", resumo_formato["cine_nulos"], "/", resumo_formato["cine_fora_padrao"])
print("CBO nulos/fora do padrão:", resumo_formato["cbo_nulos"], "/", resumo_formato["cbo_fora_padrao"])

if (
    resumo_formato["cine_nulos"] > 0
    or resumo_formato["cbo_nulos"] > 0
    or resumo_formato["cine_fora_padrao"] > 0
    or resumo_formato["cbo_fora_padrao"] > 0
):
    print("\nInconsistências de nulos/formato — linhas afetadas:")
    display(
        df_mapeamento.filter(
            F.col("codigo_cine_rotulo").isNull()
            | F.col("codigo_cbo").isNull()
            | ~F.col("codigo_cine_rotulo").rlike(PADRAO_CINE)
            | ~F.col("codigo_cbo").rlike(PADRAO_CBO)
        )
    )
    raise ValueError(
        "CSV contém inconsistências de nulos/formato — corrija a fonte antes de prosseguir."
    )

# 2) Pares duplicados — deduplicação documentada (nunca silenciosa)
df_duplicados = (
    df_mapeamento
    .groupBy("codigo_cine_rotulo", "codigo_cbo")
    .agg(F.count("*").alias("qtd_repeticoes"))
    .filter(F.col("qtd_repeticoes") > 1)
)
qtd_duplicados = df_duplicados.count()

if qtd_duplicados > 0:
    print(f"\nALERTA: {qtd_duplicados} pares duplicados — pares afetados:")
    display(df_duplicados.orderBy("codigo_cine_rotulo", "codigo_cbo"))
    df_mapeamento = df_mapeamento.dropDuplicates(["codigo_cine_rotulo", "codigo_cbo"])
    print("Deduplicação aplicada (exibida acima).")
else:
    print("Nenhum par duplicado.")

# 3) Existência nas dimensões — pares fora das dims saem da bridge de forma documentada
df_chaves_cine = spark.table(DIM_CINE).select("codigo_cine_rotulo")
df_chaves_cbo = spark.table(DIM_CBO).select("codigo_cbo")

df_pares_cine_ausente = df_mapeamento.join(df_chaves_cine, "codigo_cine_rotulo", "left_anti")
df_pares_cbo_ausente = df_mapeamento.join(df_chaves_cbo, "codigo_cbo", "left_anti")
df_pares_fora_dims = (
    df_pares_cine_ausente.unionByName(df_pares_cbo_ausente).distinct()
)

qtd_fora_dims = df_pares_fora_dims.count()

if qtd_fora_dims > 0:
    print(
        f"\nALERTA: {qtd_fora_dims} pares referenciam códigos ausentes nas dimensões "
        "— serão EXCLUÍDOS da bridge (exclusão documentada, não silenciosa):"
    )
    display(df_pares_fora_dims.orderBy("codigo_cine_rotulo", "codigo_cbo"))
    df_mapeamento_valido = (
        df_mapeamento
        .join(df_chaves_cine, "codigo_cine_rotulo", "left_semi")
        .join(df_chaves_cbo, "codigo_cbo", "left_semi")
    )
else:
    print("Todos os códigos dos pares existem nas dimensões.")
    df_mapeamento_valido = df_mapeamento

print("\nPares únicos válidos para a bridge:", df_mapeamento_valido.count())

In [0]:
# Indicadores de cobertura por CINE (inclui rótulos sem correspondência)
df_pares_cine = df_mapeamento_valido.groupBy("codigo_cine_rotulo").agg(
    F.countDistinct("codigo_cbo").alias("qtd_cbos")
)

df_cobertura_cine = (
    spark.table(DIM_CINE)
    .select("codigo_cine_rotulo", "nome_cine_rotulo")
    .join(df_pares_cine, "codigo_cine_rotulo", "left")
    .na.fill(0, subset=["qtd_cbos"])
)

resumo_cobertura_cine = df_cobertura_cine.agg(
    F.count("*").alias("rotulos_dim"),
    F.sum(F.when(F.col("qtd_cbos") > 0, 1).otherwise(0)).alias("com_correspondencia"),
    F.sum(F.when(F.col("qtd_cbos") == 0, 1).otherwise(0)).alias("sem_correspondencia"),
).collect()[0]

print("== Cobertura por CINE ==")
print("Rótulos na dim:", resumo_cobertura_cine["rotulos_dim"])
print("Com correspondência:", resumo_cobertura_cine["com_correspondencia"])
print("Sem correspondência:", resumo_cobertura_cine["sem_correspondencia"])

if resumo_cobertura_cine["sem_correspondencia"] > 0:
    print("\nRótulos CINE sem correspondência:")
    display(
        df_cobertura_cine
        .filter(F.col("qtd_cbos") == 0)
        .select("codigo_cine_rotulo", "nome_cine_rotulo")
        .orderBy("codigo_cine_rotulo")
    )

# Indicadores de cobertura por CBO (inclui ocupações sem correspondência)
df_pares_cbo = df_mapeamento_valido.groupBy("codigo_cbo").agg(
    F.countDistinct("codigo_cine_rotulo").alias("qtd_cines")
)

df_cobertura_cbo = (
    spark.table(DIM_CBO)
    .select("codigo_cbo", "nome_ocupacao")
    .join(df_pares_cbo, "codigo_cbo", "left")
    .na.fill(0, subset=["qtd_cines"])
)

resumo_cobertura_cbo = df_cobertura_cbo.agg(
    F.count("*").alias("cbos_dim"),
    F.sum(F.when(F.col("qtd_cines") > 0, 1).otherwise(0)).alias("com_correspondencia"),
    F.sum(F.when(F.col("qtd_cines") == 0, 1).otherwise(0)).alias("sem_correspondencia"),
).collect()[0]

print("\n== Cobertura por CBO ==")
print("Códigos na dim:", resumo_cobertura_cbo["cbos_dim"])
print("Com correspondência:", resumo_cobertura_cbo["com_correspondencia"])
print("Sem correspondência:", resumo_cobertura_cbo["sem_correspondencia"])

In [0]:
from pyspark.sql import functions as F


def gravar_bridge_idempotente(
    df_bridge,
    nome_tabela,
    colunas_chave,
    coluna_versao,
    ddl_create,
    versao_mapeamento,
):
    """Grava a bridge Delta de forma idempotente.

    A gravação só acontece quando:
    - a tabela ainda não existe; ou
    - o conteúdo da fonte mudou (diferença simétrica); ou
    - a versão do mapeamento mudou.
    Do contrário, a execução é um no-op e a tabela é preservada.
    """
    # --- Unicidade dos pares antes da gravação ---
    resumo = df_bridge.agg(
        F.count("*").alias("qtd_registros"),
        F.countDistinct(*colunas_chave).alias("qtd_pares"),
    ).collect()[0]

    assert resumo["qtd_registros"] == resumo["qtd_pares"], (
        f"[{nome_tabela}] Grão violado: {resumo['qtd_registros']} registros "
        f"para {resumo['qtd_pares']} pares distintos."
    )

    existe = spark.catalog.tableExists(nome_tabela)

    if existe:
        df_atual = spark.table(nome_tabela)

        colunas_negocio = list(df_bridge.columns)
        colunas_esperadas = colunas_negocio + ["dt_processamento"]
        assert list(df_atual.columns) == colunas_esperadas, (
            f"[{nome_tabela}] Schema existente diverge do esperado — "
            f"revisar manualmente antes de regravar.\n"
            f"Esperado: {colunas_esperadas}\n"
            f"Atual: {list(df_atual.columns)}"
        )

        versao_atual = df_atual.select(
            F.max(coluna_versao)
        ).collect()[0][0]

        mudancas = (
            df_bridge.select(*colunas_negocio)
            .exceptAll(df_atual.select(*colunas_negocio))
            .count()
            + df_atual.select(*colunas_negocio)
            .exceptAll(df_bridge.select(*colunas_negocio))
            .count()
        )

        if mudancas == 0 and versao_atual == versao_mapeamento:
            print(
                f"[{nome_tabela}] Fonte inalterada e versão "
                f"{versao_mapeamento} mantida — gravação ignorada."
            )
            return False

        motivos = []
        if mudancas > 0:
            motivos.append(
                f"conteúdo da fonte mudou ({mudancas} linhas diferentes)"
            )
        if versao_atual != versao_mapeamento:
            motivos.append(
                f"versão do mapeamento {versao_atual} -> {versao_mapeamento}"
            )
        print(f"[{nome_tabela}] Regravando: {'; '.join(motivos)}.")
    else:
        print(f"[{nome_tabela}] Tabela inexistente — criando pela primeira vez.")

    # --- Metadados de controle ---
    df_final = df_bridge.withColumn(
        "dt_processamento", F.current_timestamp()
    )

    if not existe:
        spark.sql(ddl_create)

    # INSERT OVERWRITE com lista explícita de colunas (alinhada ao DDL)
    lista_colunas = ", ".join(df_final.columns)
    df_final.createOrReplaceTempView("vw_bridge_gravacao")
    spark.sql(
        f"INSERT OVERWRITE {nome_tabela} "
        f"SELECT {lista_colunas} FROM vw_bridge_gravacao"
    )

    qtd_gravada = spark.table(nome_tabela).count()
    print(f"[{nome_tabela}] Gravação concluída: {qtd_gravada} pares.")
    return True

In [0]:
DDL_BRIDGE = f"""
CREATE TABLE IF NOT EXISTS {BRIDGE} (
    codigo_cine_rotulo STRING COMMENT 'Código CINE do rótulo (chave para dim_cine_brasil)',
    codigo_cbo STRING COMMENT 'Código CBO 2002 com 6 dígitos (chave para dim_cbo_ocupacao)',
    fonte_mapeamento STRING COMMENT 'Fonte da correspondência (CSV do MEC)',
    versao_mapeamento STRING COMMENT 'Versão do mapeamento recebido',
    dt_processamento TIMESTAMP COMMENT 'Data e hora da gravação da bridge'
)
USING DELTA
COMMENT 'Bridge CINE × CBO: correspondência entre rótulos CINE Brasil e ocupações CBO 2002'
"""

df_bridge_final = df_mapeamento_valido.select(
    "codigo_cine_rotulo",
    "codigo_cbo",
    F.lit(FONTE_MAPEAMENTO).alias("fonte_mapeamento"),
    F.lit(VERSAO_MAPEAMENTO).alias("versao_mapeamento"),
)

print("Tabela-alvo:", BRIDGE)
print("Pares a gravar:", df_bridge_final.count())

gravou_bridge = gravar_bridge_idempotente(
    df_bridge=df_bridge_final,
    nome_tabela=BRIDGE,
    colunas_chave=["codigo_cine_rotulo", "codigo_cbo"],
    coluna_versao="versao_mapeamento",
    ddl_create=DDL_BRIDGE,
    versao_mapeamento=VERSAO_MAPEAMENTO,
)

In [0]:
from pyspark.sql import functions as F

df_gold_bridge = spark.table(BRIDGE)

resumo_bridge = df_gold_bridge.agg(
    F.count("*").alias("qtd_pares"),
    F.countDistinct("codigo_cine_rotulo", "codigo_cbo").alias("qtd_pares_unicos"),
    F.countDistinct("codigo_cine_rotulo").alias("cines_distintos"),
    F.countDistinct("codigo_cbo").alias("cbos_distintos"),
    F.max("versao_mapeamento").alias("versao"),
    F.max("dt_processamento").alias("dt_ultimo_processamento"),
).collect()[0]

print("Pares persistidos:", resumo_bridge["qtd_pares"])
print("Pares únicos:", resumo_bridge["qtd_pares_unicos"])
print("Rótulos CINE distintos:", resumo_bridge["cines_distintos"])
print("Códigos CBO distintos:", resumo_bridge["cbos_distintos"])
print("Versão do mapeamento:", resumo_bridge["versao"])
print("Último processamento:", resumo_bridge["dt_ultimo_processamento"])

assert resumo_bridge["qtd_pares"] == resumo_bridge["qtd_pares_unicos"], (
    f"[{BRIDGE}] Unicidade de pares violada na tabela persistida"
)

# Integridade: todo código da bridge existe nas dimensões
violacoes_cine = df_gold_bridge.join(
    spark.table(DIM_CINE).select("codigo_cine_rotulo"),
    "codigo_cine_rotulo",
    "left_anti",
).count()

violacoes_cbo = df_gold_bridge.join(
    spark.table(DIM_CBO).select("codigo_cbo"),
    "codigo_cbo",
    "left_anti",
).count()

print("Pares com CINE ausente na dim:", violacoes_cine)
print("Pares com CBO ausente na dim:", violacoes_cbo)

assert violacoes_cine == 0 and violacoes_cbo == 0, (
    f"[{BRIDGE}] Integridade com as dimensões violada"
)

display(df_gold_bridge.orderBy("codigo_cine_rotulo", "codigo_cbo").limit(20))

## Observações

* O CSV de correspondência CINE–CBO do MEC **ainda não foi recebido**. Enquanto ausente, o notebook encerra na célula de disponibilidade com status `AGUARDANDO_FONTE` e **não cria a bridge** — nenhuma correspondência fictícia é gerada.
* Para processar: coloque o CSV em `/Volumes/workspace/bronze/raw_cine_cbo` (ou aponte os parâmetros `caminho_diretorio_csv`/`arquivo_csv` para o local correto) e reexecute o notebook — as demais células executam automaticamente na ordem.
* O CSV é lido com todas as colunas como STRING (zeros à esquerda preservados), separador configurável via `separador_csv` (padrão `;`).
* Inconsistências (nulos, formato, códigos fora das dimensões) são sempre exibidas antes de qualquer ação; pares com códigos ausentes nas dimensões só saem da bridge com exclusão **documentada**.
* Dimensões e Golds existentes não são alteradas por este notebook.
